In [24]:
import Gmsh: gmsh
using Gridap, GridapGmsh

In [25]:
function create_square_model(h)
    gmsh.initialize()
    gmsh.model.add("unit_square");
    gmsh.model.geo.addPoint(0.0, 0.0, 0.0, h, 1); # last argument is optional identifier, unique per dimension
    gmsh.model.geo.addPoint(1.0, 0.0, 0.0, h, 2);
    gmsh.model.geo.addPoint(1.0, 1.0, 0.0, h, 3);
    gmsh.model.geo.addPoint(0.0, 1.0, 0.0, h, 4);
    gmsh.model.geo.addLine(1, 2, 1); 
    gmsh.model.geo.addLine(2, 3, 2); # line 2 goes from point 2 to point 3
    gmsh.model.geo.addLine(3, 4, 3);
    gmsh.model.geo.addLine(4, 1, 4);
    
    gmsh.model.geo.addCurveLoop([1, 2, 3, 4], 1);

    gmsh.model.geo.addPlaneSurface([1], 1);

    gmsh.model.geo.synchronize();

    # Define physical groups without the string argument
    edges_tag = gmsh.model.addPhysicalGroup(1, [1, 2, 3, 4])   # edges
    corners_tag = gmsh.model.addPhysicalGroup(0, [1, 2, 3, 4]) # corners
    domain_tag = gmsh.model.addPhysicalGroup(2, [1])           # surface
    
    # Set names for the physical groups
    gmsh.model.setPhysicalName(1, edges_tag, "boundary")
    gmsh.model.setPhysicalName(0, corners_tag, "boundary")
    gmsh.model.setPhysicalName(2, domain_tag, "domain")

    gmsh.model.mesh.generate(2); # dimension is 2
    
    model = GmshDiscreteModel(gmsh);
    gmsh.finalize();
    return model
end


create_square_model (generic function with 1 method)

In [26]:
h = 0.05; # target mesh size
model = create_square_model(h) # fix above function above using the tutorial
order = 1
reffe = ReferenceFE(lagrangian, Float64, order)
V = TestFESpace(model, reffe, conformity=:H1, dirichlet_tags="boundary")
U = TrialFESpace(V, 0.0)


Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 30%] Meshing curve 2 (Line)
Info    : [ 60%] Meshing curve 3 (Line)
Info    : [ 80%] Meshing curve 4 (Line)
Info    : Done meshing 1D (Wall 0.00121131s, CPU 0.001271s)
Info    : Meshing 2D...
Info    : Meshing surface 1 (Plane, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.0239642s, CPU 0.023947s)
Info    : 513 nodes 1028 elements


TrialFESpace()

In [27]:
Ω = Triangulation(model)
dΩ = Measure(Ω, 2 * order)

GenericMeasure()

In [28]:
n=5
Δt, T = 2.0^(-n), 1.0 # time step, final time
a0 = 0.05 # diffusion coefficient
num_steps = Int(T / Δt)

32

In [29]:
f(t, x) = sin(π * x[1]) * sin(2 * π * x[2]) * (5 * a0 * π^2 * (2 - cos(t)) + sin(t))
f(t) = x -> f(t,x)

f (generic function with 2 methods)

In [31]:
u_exact(t, x) = sin(pi * x[1]) * sin(2pi * x[2]) * (2 - cos(t)) #int from 0 to t # fill in the rests
u₀ = x -> sin(pi * x[1]) * sin(2pi * x[2]) # fill in initial condition based on u_exact

#53 (generic function with 1 method)

In [32]:
if !isdir("tmp") mkdir("tmp") end # for storing solution
# assemble the left hand side:
a(u, v) =  ∫( ∇(v)⋅∇(u) ) * dΩ # fill in the rest
aa = assemble_matrix(a, U, V)

433×433 SparseArrays.SparseMatrixCSC{Float64, Int64} with 2873 stored entries:
⎡⠑⢄⠈⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⡖⡏⠹⣤⡤⣀⣁⠀⠒⠀⠀⠀⠀⠐⢶⣔⡲⢿⡤⡠⎤
⎢⠂⠀⠱⣦⡀⠀⢀⠀⠠⠀⠀⡀⡄⠀⠀⢀⠀⠀⡀⠀⠈⠊⠈⠊⠐⠂⠆⠀⠉⣠⠀⠠⠰⠁⠂⠉⠂⠓⠑⠉⎥
⎢⠀⠀⠀⠈⠻⣦⡈⠄⠀⠀⠀⠀⠀⢀⠄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢰⡀⠘⡵⠈⠆⠙⠂⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠐⠂⠌⠱⣦⡐⠬⢉⠀⡄⠈⠄⣰⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠹⢰⠺⣦⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠂⠀⠀⡐⡌⢿⣷⠠⣆⡠⢄⡀⢁⠀⠀⠀⠀⠀⢂⠀⠂⠀⠀⠃⣰⢰⣐⢽⠪⠔⠀⠀⠠⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠠⠀⠀⠃⠐⠠⢦⠟⣥⡺⢕⠲⣠⡇⠀⠀⠀⠀⠀⠤⠀⠀⠀⠧⠃⠐⢄⡃⠄⠖⢢⠀⠀⠀⠀⠄⠀⎥
⎢⠀⠀⠀⠉⠀⢀⡀⠉⠀⢎⢞⢎⡿⣯⣹⣁⢂⠲⡓⠠⠀⠀⠁⡀⠀⠄⣂⢁⡠⠣⡕⢎⠄⠁⠀⠀⢠⠀⠔⠄⎥
⎢⠀⠀⠀⢀⠀⠁⢀⣡⠄⢈⠘⣢⠗⢺⠕⢅⠨⠂⡈⡕⢂⠄⡤⡄⠒⡂⢤⣁⢨⣧⣲⡻⡲⢘⡨⠀⠰⠀⠄⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠉⢨⡐⠢⠂⡱⣮⣥⢂⡤⣎⠅⠄⠀⠀⡀⠉⠒⢀⢣⠂⢁⠼⠒⠍⡀⡀⠨⡄⎥
⎢⠀⢀⠀⠈⠀⠀⠀⠀⠀⠀⠀⠀⠙⡈⢆⠬⠡⢛⠱⢆⡪⠅⢁⡁⠀⠀⠀⠌⠈⠪⢘⡔⠿⢶⡤⢒⡬⣁⠐⣂⎥
⎢⡼⠭⡢⠀⠀⠀⠀⠀⠠⢀⠀⠀⠀⠀⠈⠔⡠⢯⠎⠎⠑⢄⣀⠄⠄⠂⡀⡠⠢⠰⢐⢡⠘⡀⠶⠉⠱⢷⠀⢑⎥
⎢⠓⣦⡢⠀⠀⠀⠀⠀⠠⠀⠀⠃⠁⠠⠀⠯⠁⠅⠅⠰⠀⠜⠟⢅⠀⠀⠆⠰⠤⠒⠐⠀⠄⠁⢀⠋⠤⢖⡘⠥⎥
⎢⠀⢫⠰⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠄⠸⠠⠀⠀⠀⠀⠠⠁⠀⠀⠻⣦⣠⠀⡆⠀⠄⠘⠈⠀⠀⠀⠋⢴⡣⡀⎥
⎢⠁⠘⠈⠁⠐⠲⣄⡀⢉⣠⠭⠃⠌⢘⠄⢳⡄⠈⡀⠄⠀⡨⢈⡁⠀⠚⡛⢌⣑⢄⢜⡺⡬⠓⠀⠀⢁⠀⠁⠂⎥
⎢⠘⠀⠃⣠⢖⡤⣰⡒⢐⢲⠐⢄⠤⡊⠦⣶⠘⢀⡢⡀⢈⡂⢠⠃⠈⠉⠑⢜⢑⢔⢱⡵⠤⠲⠀⠀⡉⢁⡁⠀⎥
⎢⠀⠀⠀⡀⠢⠄⠈⠛⡳⡓⠉⠌⡱⢍⣼⡺⠩⠒⢒⠴⠔⣐⠐⠀⣀⠁⣲⡱⢕⡶⢑⣴⣣⡄⠐⠈⠠⠐⠀⠤⎥
⎢⢀⠀⠔⠂⠳⠀⠀⠀⠐⠁⠸⣁⠄⠁⣘⢊⣁⡔⢻⣇⠒⠠⠄⠁⠂⠀⢦⠋⢠⡃⠉⠾⡻⣮⡂⠘⠘⠀⠀⠁⎥
⎢⢘⢷⡌⠀⠀⠀⠀⠀⠀⡀⠀⠀⠀⠀⠂⠊⡜⠄⢠⢋⡜⠃⡤⠐⠀⠀⠀⠀⠀⠀⡐⠀⣈⠈⠛⢄⠀⠓⠢⡄⎥
⎢⣼⣎⢬⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠒⠐⠂⠀⠨⠆⢫⢵⣆⢠⢇⢋⣄⠁⠐⠇⢈⢀⠂⠒⠀⢤⠀⠟⢅⠋⢹⎥
⎣⠀⡫⡕⠀⠀⠀⠀⠀⠀⠀⠀⠁⠐⠅⠀⠁⠂⠦⠰⢠⢄⢀⠖⡌⠉⠪⠡⠀⠁⠈⠀⡄⠄⠀⠈⠦⣏⣀⠵⢇⎦

In [34]:
createpvd("heat") do pvd
    uh = interpolate_everywhere(u₀, U) # initial condition
    pvd[0] = createvtk(Ω, "tmp/heat_0.vtu", cellfields=["uh" => uh, "e" => x -> 0.0])
    e = nothing # to make the error accessible after the loop
    for n in 1:num_steps # start time-stepping
        tₙ = n * Δt
        tₙ₋₁ = (n-1) * Δt
        
        # assemble the right hand side
        b(v) = ∫( v * f ) * dΩ # fill in the rest
        bb = assemble_vector(b, V)
        uh = FEFunction(U, aa \ bb) # solve for next value
        
        # save numerical solution and error
        vtufilename = "tmp/heat_$n.vtu"
        u_exact_n = x -> u_exact(tₙ, x)
        e = u_exact_n - uh
        pvd[tₙ] = createvtk(Ω, vtufilename, cellfields=["uh" => uh, "e" => e])
    end
    el2 = sqrt(sum( ∫( e*e )*dΩ )) # fill in the rest
    println("L2 error norm: $el2")
end


LoadError: MethodError: no method matching zero(::Type{var"#51#52"{VectorValue{2, Float64}}})
The function `zero` exists, but no method is defined for this combination of argument types.

[0mClosest candidates are:
[0m  zero([91m::Type{Union{}}[39m, Any...)
[0m[90m   @[39m [90mBase[39m [90m[4mnumber.jl:310[24m[39m
[0m  zero([91m::Type{Pkg.Resolve.VersionWeight}[39m)
[0m[90m   @[39m [36mPkg[39m [90m/usr/local/julia/share/julia/stdlib/v1.11/Pkg/src/Resolve/[39m[90m[4mversionweights.jl:15[24m[39m
[0m  zero([91m::Type{Missing}[39m)
[0m[90m   @[39m [90mBase[39m [90m[4mmissing.jl:104[24m[39m
[0m  ...
